In [ ]:
%matplotlib widget
import scipp as sc
import plopp as pp
import scippneutron as scn
import scippnexus as snx
import h5py
from pathlib import Path
import numpy as np

In [ ]:
from ess.spectroscopy.indirect import bifrost
from bifrost2409.config import POOCH_DATA_DIR, INTERIM_DATA_DIR
from bifrost2409.dataset import download_datafiles

In [ ]:
datafile = "20240914/BIFROST_20240914T053723.h5"
download_datafiles([datafile])

In [ ]:
targets = [
    'wavelength_monitor',
    'norm_events',
    'triplet_events',
]
target_files = {target: INTERIM_DATA_DIR / f'{Path(datafile).stem}_{target}.h5' for target in targets}
if all(file.exists() for file in target_files.values()):
    from scipp.io import load_hdf5
    from loguru import logger
    from rich.pretty import pretty_repr
    objects = {target: load_hdf5(file) for target, file in target_files.items()}
    logger.info(f'Loaded objects {pretty_repr(target_files)}')
else:
    data = bifrost(POOCH_DATA_DIR / datafile, is_simulated=True)
    objects = {target: data[target] for target in targets}
    for target in targets:
        objects[target].save_hdf5(target_files[target])

In [ ]:
trip = objects['triplet_events']

In [ ]:
events, monitor = [objects[x] for x in ('norm_events', 'wavelength_monitor')]

In [ ]:
events

In [ ]:
# from ess.spectroscopy.indirect import bifrost_to_nxspe
# nxspe_output = INTERIM_DATA_DIR / 'nxspe' / f'{Path(datafile).stem}'
# nxspe_files = bifrost_to_nxspe(events=events, output=nxspe_output)

In [ ]:
from ess.spectroscopy.indirect.sqw import to_sqw

In [ ]:
# Make 'setting' an explicit coordinate, this is used in place of 'run number' for SQW
events.coords['setting'] = sc.arange('setting', events.sizes['setting'], None)
events

In [ ]:
# All BIFROST Q==(Qx, 0, Qz) so Qy is not represented but the generic to_sqw requires an explicit representation
events.bins.coords['table_momentum_y'] = 0 * events.bins.coords['table_momentum_x']

In [ ]:
# All BIFROST kf-hats are in the horizontal plane, so phi==0 is implicit. The Generig to_sqw requires this to be explicit
events.coords['phi'] = 0 * events.coords['theta']

In [ ]:
qe_names = ('table_momentum_x', 'table_momentum_y', 'table_momentum_z', 'energy_transfer')
limits = {x: (sc.min(events.bins.coords[x]), sc.max(events.bins.coords[x])) for x in qe_names}
qe_bins = {x: sc.linspace(start=y[0], stop=y[1], num=51, dim=x) for x, y in limits.items()}
qe_bins['table_momentum_y'] = sc.array(values=[-0.1, 0.1], dims=['table_momentum_y'], unit='1/angstrom')

In [ ]:
qe_bins

In [ ]:
b = to_sqw(events, '', qe_bins['table_momentum_x'], qe_bins['table_momentum_y'], qe_bins['table_momentum_z'], qe_bins['energy_transfer'])

In [ ]:
b